In [6]:
# 1. 挂载 Google Drive (必须先做这一步，否则找不到你的视频)
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

# 2. 准备工作目录
# 我们在 Colab 的临时空间 /content 下下载代码，这样不会弄乱你的云盘
%cd /content
if not os.path.exists("Grounded-SAM-2"):
    !git clone https://github.com/IDEA-Research/Grounded-SAM-2.git

%cd Grounded-SAM-2

# 3. 安装 SAM 2 和 Grounding DINO
# 安装 SAM 2
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
# 安装 Grounding DINO (需要编译 CUDA 算子，可能需要 1-3 分钟)
!pip install -q -e grounding_dino

# 4. 安装可视化工具
!pip install -q supervision opencv-python pycocotools matplotlib onnxruntime onnx

print("✅ 环境安装完成，云盘已挂载！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/Grounded-SAM-2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
✅ 环境安装完成，云盘已挂载！


In [7]:
# cell 2: 下载模型权重
import os

# 创建权重目录
!mkdir -p gdino_checkpoints
!mkdir -p checkpoints

# 1. 下载 Grounding DINO 权重
if not os.path.exists("gdino_checkpoints/groundingdino_swint_ogc.pth"):
    print("正在下载 Grounding DINO 权重...")
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -P gdino_checkpoints/

# 2. 下载 SAM 2 (Hiera-Large) 权重 - 这是效果最好的版本
if not os.path.exists("checkpoints/sam2_hiera_large.pt"):
    print("正在下载 SAM 2 权重...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt -P checkpoints/

print("✅ 模型权重下载完成！")

✅ 模型权重下载完成！


In [8]:
# Cell 3: 导入库 & 定义辅助函数
import torch
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import supervision as sv
import torchvision.ops.boxes as bops

# 导入模型相关
from groundingdino.util.inference import load_model, load_image, predict
from sam2.build_sam import build_sam2_video_predictor

# 检查 GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# 定义视频拆帧函数 (SAM 2 需要读取帧文件夹)
def split_video_to_frames(video_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    print(f"正在拆解视频: {video_path}")
    print(f"目标文件夹: {output_dir}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # 保存为 00000.jpg
        frame_name = f"{frame_count:05d}.jpg"
        cv2.imwrite(os.path.join(output_dir, frame_name), frame)
        frame_count += 1

    cap.release()
    print(f"✅ 拆解完成，共 {frame_count} 帧。")
    return frame_count

Using device: cuda


In [11]:
# Cell 4: 配置 & 加载模型

# ==================== 你的配置 ====================
# 输入视频路径
VIDEO_PATH = "/content/drive/MyDrive/sam2/notebooks/videos/bunny.mp4"

# 输出视频路径
OUTPUT_VIDEO_PATH = "/content/drive/MyDrive/sam2/notebooks/videos/bunny_result.mp4"

# 提示词 (根据你的视频内容修改，bunny 建议用 rabbit)
TEXT_PROMPT = "stuffed animal"

# 临时帧文件夹 (存放在 Colab 本地，不占用云盘空间，跑完会消失)
FRAME_DIR = "/content/temp_bunny_frames"
# =================================================

# 检查输入视频是否存在
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"❌ 错误：找不到文件 {VIDEO_PATH}，请检查路径是否正确！")

# 1. 拆解视频
split_video_to_frames(VIDEO_PATH, FRAME_DIR)

# 2. 加载模型配置
GDINO_CONFIG = "grounding_dino/groundingdino/config/GroundingDINO_SwinT_OGC.py"
GDINO_CHECKPOINT = "gdino_checkpoints/groundingdino_swint_ogc.pth"
SAM2_CONFIG = "configs/sam2/sam2_hiera_l.yaml"
SAM2_CHECKPOINT = "checkpoints/sam2_hiera_large.pt"

# 3. 加载模型
print("正在加载模型...")
gdino_model = load_model(GDINO_CONFIG, GDINO_CHECKPOINT)
sam2_video_predictor = build_sam2_video_predictor(SAM2_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
print("✅ 模型加载完毕！")

正在拆解视频: /content/drive/MyDrive/sam2/notebooks/videos/bunny.mp4
目标文件夹: /content/temp_bunny_frames
✅ 拆解完成，共 102 帧。
正在加载模型...
final text_encoder_type: bert-base-uncased
✅ 模型加载完毕！


In [17]:
import os
import torch
import numpy as np
import cv2
import torchvision
import supervision as sv
import torchvision.ops.boxes as bops
from tqdm import tqdm
from groundingdino.util.inference import load_image, predict

# ==================== 1. 关键变量定义 (修复报错) ====================
TEXT_PROMPT = "stuffed animal"  # 提示词
BOX_THRESHOLD = 0.35          # 检测框阈值
TEXT_THRESHOLD = 0.25         # 文本匹配阈值

# 路径配置 (请确保这些路径和你环境里的一致)
VIDEO_PATH = "/content/drive/MyDrive/sam2/notebooks/videos/bunny.mp4"
OUTPUT_VIDEO_PATH = "/content/drive/MyDrive/sam2/notebooks/videos/bunny_result_bbox.mp4" # 改个名避免覆盖
FRAME_DIR = "/content/temp_bunny_frames"
# ==================================================================

# --- 2. 首帧检测 (Grounding DINO) + 强制 Top-K 过滤 ---
print(f"Step 1: 正在首帧检测提示词 '{TEXT_PROMPT}' ...")
first_frame_path = os.path.join(FRAME_DIR, "00000.jpg")
image_source, image = load_image(first_frame_path)

# 运行检测
boxes, logits, phrases = predict(
    model=gdino_model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD
)

# 坐标转换 cxcywh -> xyxy
h, w, _ = image_source.shape
boxes_xyxy = boxes * torch.Tensor([w, h, w, h])
boxes_xyxy = bops.box_convert(boxes=boxes_xyxy, in_fmt="cxcywh", out_fmt="xyxy")

print(f"🔍 原始检测: 发现 {len(boxes_xyxy)} 个目标")

# ==================== 核心修改：强制只留前 2 名 ====================
# 1. 先做一次 NMS 去重 (防止同一个兔子身上有两个框)
nms_threshold = 0.5
nms_indices = torchvision.ops.nms(boxes_xyxy, logits, nms_threshold)
boxes_xyxy = boxes_xyxy[nms_indices]
logits = logits[nms_indices]
phrases = [phrases[i] for i in nms_indices]

# 2. 强制 Top-K：如果你确定只有 2 个目标，就只取分数最高的 2 个
MAX_OBJECTS = 2  # <--- 这里设置为你想保留的最大数量

if len(logits) > MAX_OBJECTS:
    print(f"⚠️ 目标过多 ({len(logits)}个)，强制保留分数最高的 {MAX_OBJECTS} 个...")
    # 获取分数最高的索引
    topk_values, topk_indices = torch.topk(logits, k=MAX_OBJECTS)

    # 过滤数据
    boxes_xyxy = boxes_xyxy[topk_indices]
    logits = logits[topk_indices]
    phrases = [phrases[i] for i in topk_indices]

# ===============================================================

boxes_xyxy = boxes_xyxy.numpy()

if len(boxes_xyxy) == 0:
    raise ValueError(f"⚠️ 警告：检测后没有剩余目标！请尝试降低 BOX_THRESHOLD。")
else:
    print(f"✅ 最终保留: {len(boxes_xyxy)} 个目标。")
    print(f"   置信度分数: {logits.numpy()}")

# --- 3. 初始化 SAM 2 并注入提示 ---
print("Step 2: 初始化 SAM 2 视频状态...")
inference_state = sam2_video_predictor.init_state(video_path=FRAME_DIR)
sam2_video_predictor.reset_state(inference_state)

print("Step 3: 注入首帧 Box 提示...")
for i, box in enumerate(boxes_xyxy):
    obj_id = i + 1
    # 关键：只在第 0 帧添加提示
    _, out_obj_ids, out_mask_logits = sam2_video_predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=obj_id,
        box=box
    )

# --- 4. 视频传播 (Video Propagation) ---
print("Step 4: 开始视频传播 (Tracking)...")
video_segments = {}

# 自动处理后续所有帧
for out_frame_idx, out_obj_ids, out_mask_logits in sam2_video_predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        "out_obj_ids": out_obj_ids,
        "out_mask_logits": out_mask_logits
    }

# --- 5. 合成并保存视频 (增加了 BBox) ---
print(f"Step 5: 正在合成视频到 {OUTPUT_VIDEO_PATH} ...")

# 初始化注释器
# 1. Mask 注释器 (画遮罩)
mask_annotator = sv.MaskAnnotator(
    color_lookup=sv.ColorLookup.INDEX
)
# 2. Box 注释器 (画边框)
box_annotator = sv.BoxAnnotator(
    color_lookup=sv.ColorLookup.INDEX,
    thickness=2  # 边框粗细
)

frame_names = sorted([p for p in os.listdir(FRAME_DIR) if p.endswith(".jpg")])

# 准备写入视频
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, 30, (w, h))

for frame_idx in tqdm(range(len(frame_names))):
    frame_path = os.path.join(FRAME_DIR, frame_names[frame_idx])
    frame = cv2.imread(frame_path)

    if frame_idx in video_segments:
        out_obj_ids = video_segments[frame_idx]["out_obj_ids"]
        out_mask_logits = video_segments[frame_idx]["out_mask_logits"]

        # 转换 Mask
        masks = (out_mask_logits > 0.0).cpu().numpy().squeeze(1)

        if len(out_obj_ids) > 0:
            # 创建 Detections 对象
            # sv.mask_to_xyxy 会自动根据 Mask 计算出当前的 BBox
            detections = sv.Detections(
                xyxy=sv.mask_to_xyxy(masks),
                mask=masks,
                class_id=np.array(out_obj_ids)
            )

            # 1. 先画 Mask
            frame = mask_annotator.annotate(scene=frame, detections=detections)
            # 2. 再画 Box (叠加在 Mask 之上)
            frame = box_annotator.annotate(scene=frame, detections=detections)

    out_video.write(frame)

out_video.release()
print(f"🎉 恭喜！带边框的视频处理完成，已保存至云盘：\n{OUTPUT_VIDEO_PATH}")

Step 1: 正在首帧检测提示词 'stuffed animal' ...
🔍 原始检测: 发现 3 个目标
⚠️ 目标过多 (3个)，强制保留分数最高的 2 个...
✅ 最终保留: 2 个目标。
   置信度分数: [0.6755754  0.53221303]
Step 2: 初始化 SAM 2 视频状态...


frame loading (JPEG): 100%|██████████| 102/102 [00:03<00:00, 26.96it/s]


Step 3: 注入首帧 Box 提示...
Step 4: 开始视频传播 (Tracking)...


propagate in video: 100%|██████████| 102/102 [02:29<00:00,  1.47s/it]


Step 5: 正在合成视频到 /content/drive/MyDrive/sam2/notebooks/videos/bunny_result_bbox.mp4 ...


100%|██████████| 102/102 [00:04<00:00, 25.22it/s]

🎉 恭喜！带边框的视频处理完成，已保存至云盘：
/content/drive/MyDrive/sam2/notebooks/videos/bunny_result_bbox.mp4
